# Phase 3: MuRIL + IndicBERT fine-tuning (Hindi)

Owner: Shreyank Sthavaramath. Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 3.

Fine-tune **MuRIL** (primary) and **IndicBERT** (comparison) for Hindi NER with
HuggingFace `AutoModelForTokenClassification` + `Trainer`. The subword
label-alignment step is the #1 NER bug (see
[docs/05-pitfalls.md](../docs/05-pitfalls.md)); it gets its own section and a
self-test below before any model is trained on it.

**The training size is found from the data, not fixed up front.** Train at an
increasing sequence of subset sizes, score Hindi gold F1 after each, log every
point as its own row in the results sheet, and stop adding data once the
F1-vs-size curve flattens. That size becomes the working subset for the rest
of the project.

**Gate:** both models scored on Hindi gold; MuRIL clearly beats the
BiLSTM-CRF baseline (**0.8038**, Phase 2); the scaling curve is logged and a
working subset size is chosen from it. *If MuRIL does not beat the baseline,
suspect the label alignment first.*

**How to run (Kaggle)**
1. Commit and push this notebook to `phase3-muril-indicbert`, because the
   run clones that branch and records its commit.
2. Import it on Kaggle with **GPU T4 x2** and **Internet** on, and attach the
   `GH_TOKEN` (read-only) and `HF_TOKEN` (write) Secrets.
3. With `DEBUG = True`, use **Save & Run All**. This checks cloning, login,
   training, upload, reload, and scoring on ~80 sentences. Its scores are
   meaningless, so do not keep its `results.csv`.
4. Set `DEBUG = False` and use **Save & Run All** again (about 4 hours). This
   one run does the 10k diagnosis, scaling curve, extra seeds, size choice,
   IndicBERT comparison, and handoff verification.
5. Download `NaamRekha/results/results.csv` and
   `NaamRekha/results/phase3_scaling_curve.png` from the version's Output tab,
   plus the executed notebook. Do not edit their contents.
6. Check the gate with `uv run python scripts/phase3_gate.py` (from the repo
   root, locally -- it only reads `results/results.csv` and this notebook,
   no GPU needed).

In [ ]:
# Pin versions (docs/04-compute.md). Colab/Kaggle ship their own torch;
# transformers/datasets/accelerate/seqeval/sentencepiece are pinned here so
# runs are reproducible across machines.
%pip install -q transformers==5.22.1 datasets==5.1.0 accelerate==1.15.0 seqeval==1.2.2 sentencepiece==0.2.2

In [ ]:
import base64
import os
import subprocess
import sys

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"
REPO_BRANCH = "phase3-muril-indicbert"


def get_secret(name):
    # Kaggle Secret, then Colab Secret, then environment variable.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook. Check for a repo file, not
# just src/, because Kaggle has its own unrelated /kaggle/src folder.
if os.path.isfile("../src/data.py"):
    REPO_ROOT = os.path.abspath("..")
elif os.path.isfile("src/data.py"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("NaamRekha")
    if not os.path.isdir(REPO_ROOT):
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        token = get_secret("GH_TOKEN")
        if token:
            # Passed through the environment so it is never saved in .git/config
            # (Kaggle keeps that folder as output) or printed if the clone fails.
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            env.update(GIT_CONFIG_COUNT="1",
                       GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
                       check=True, env=env)
sys.path.insert(0, REPO_ROOT)

# indic-bert is gated, and every run publishes its checkpoint to the Hub.
hf_token = get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_ROOT, "rev-parse", "--short", "HEAD"], text=True).strip()

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution

print("repo:", REPO_ROOT, "| commit:", GIT_COMMIT)
print("labels:", LABELS)


## Config

Every knob is in this one cell. Each `train_run()` call records its settings
in the results-row `notes` automatically.

In [ ]:
SEED = 42
LANG = "hi"

DEBUG = True    # True: ~80 sentences, 1 epoch (pipeline check). False: real runs on Kaggle.

# Kaggle's dual-T4 sessions otherwise wrap the model in DataParallel, silently
# doubling the effective batch size below. Force single-GPU so BATCH_SIZE is
# the real one (must be set before torch is imported/initializes CUDA).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# MuRIL is the primary model (roadmap). IndicBERT is the comparison model.
MODELS = {
    "MuRIL": "google/muril-base-cased",
    "IndicBERT": "ai4bharat/indic-bert",
}

MAX_LENGTH = 128      # subword tokens per sentence (docs/04-compute.md: NER sentences are short)
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LR = 2e-5
EPOCHS = 3

# Scaling ladder (roadmap: "10k, then roughly triple each step"). 900k was
# never completed in a Kaggle session, so it's dropped from the ladder.
SUBSET_SIZES = [10_000, 30_000, 100_000, 300_000]

# Size choice: repeat these sizes on the extra seeds, then pick the smallest size
# with no collapsed seed whose mean validation F1 is within SIZE_TOLERANCE of the best.
SEEDED_SIZES = [30_000, 100_000]
EXTRA_SEEDS = [43, 44]
SIZE_TOLERANCE = 0.01
COLLAPSE_F1 = 0.1     # validation F1 below this counts as a collapsed run

HF_PRIVATE = True     # visibility of the checkpoint repos created on the Hub

if DEBUG:
    SUBSET_SIZES = [80]
    EPOCHS = 1
    BATCH_SIZE = 8



In [ ]:
def pick_storage() -> str:
    # Persistent storage for checkpoints: Kaggle working dir, Google Drive, or local checkpoints/.
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/naamrekha"
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/NaamRekha"
    except ImportError:
        return os.path.join(REPO_ROOT, "checkpoints")


STORAGE = pick_storage()
os.makedirs(os.path.join(STORAGE, "phase3"), exist_ok=True)
print("checkpoints ->", os.path.join(STORAGE, "phase3"))

In [ ]:
import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device, "| GPU count:", torch.cuda.device_count())
if device.type == "cpu":
    print("WARNING: no GPU. Fine for DEBUG, too slow for the real scaling runs -- use Kaggle.")
if not DEBUG and torch.cuda.device_count() != 1:
    raise RuntimeError(f"real runs need exactly one visible GPU, found {torch.cuda.device_count()}")



## Data

Train on **silver**, evaluate on the full **gold** test set, per
[docs/03-conventions.md](../docs/03-conventions.md). `make_subset` draws a
seeded random subset (same recipe as Phase 2) and `entity_report` checks it
is not mostly `O`, per the subset rule in that doc.

In [ ]:
train_full = load_split(LANG, "train")
val = load_split(LANG, "val")
test = load_split(LANG, "test")
print(f"train {len(train_full):,} (full silver) | val {len(val):,} | test {len(test):,} (gold)")


def make_subset(rows, n, seed=SEED):
    n = min(n, len(rows))
    idx = sorted(random.Random(seed).sample(range(len(rows)), n))
    return [rows[i] for i in idx]


def entity_report(rows, name):
    counts = tag_distribution(rows)
    total = sum(counts.values())
    ents = {t: counts.get(f"B-{t}", 0) for t in ("PER", "ORG", "LOC")}
    with_ent = sum(any(t != "O" for t in tags) for _, tags in rows)
    print(f"{name}: entities {ents} | O = {counts['O'] / total:.1%} of tokens | "
          f"{with_ent / len(rows):.1%} of sentences have >=1 entity")


entity_report(test, "gold test (fixed, never subset)")

## Subword label alignment

The #1 NER bug with BERT-family models: each tokenizer splits a word into
one or more subword pieces, but we have one label per **word**. The rule
(docs/05-pitfalls.md): label only the first subword of each word with that
word's tag; every continuation subword and every special token (`[CLS]`,
`[SEP]`, padding) gets `-100` so the loss ignores it. `word_ids()` is what
makes this mapping possible. IndicBERT's SentencePiece tokenizer can emit a
standalone `▁` as a word's first piece, which holds no text, so the label goes
on the first real piece after it.

A self-test below checks this on a real training sentence *before* any
model is trained on it.

In [ ]:
from datasets import Dataset


def label_positions(word_ids, tokens):
    # word -> position of the subword that carries its label: the first piece,
    # skipping a standalone SentencePiece "▁" when a real piece follows it.
    positions = {}
    for pos, wid in enumerate(word_ids):
        if wid is not None and (wid not in positions or tokens[positions[wid]] == "▁"):
            positions[wid] = pos
    return positions


def tokenize_and_align_labels(batch, tokenizer, max_length=MAX_LENGTH):
    enc = tokenizer(batch["words"], is_split_into_words=True, truncation=True, max_length=max_length)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labeled = set(label_positions(word_ids, enc.tokens(batch_index=i)).values())
        label_ids = []
        for pos, wid in enumerate(word_ids):
            if wid is None:
                label_ids.append(-100)          # [CLS] / [SEP] / padding
            elif pos in labeled:
                label_ids.append(label2id[tags[wid]])   # labeled subword of this word
            else:
                label_ids.append(-100)          # other subword
        all_labels.append(label_ids)
    enc["labels"] = all_labels
    return enc


def to_hf_dataset(rows, tokenizer, max_length=MAX_LENGTH):
    ds = Dataset.from_dict({"words": [r[0] for r in rows], "tags": [r[1] for r in rows]})
    return ds.map(lambda b: tokenize_and_align_labels(b, tokenizer, max_length),
                  batched=True, remove_columns=["words", "tags"])

In [ ]:
from transformers import AutoTokenizer

for _model_key, _checkpoint in MODELS.items():
    _tok = AutoTokenizer.from_pretrained(_checkpoint)
    _sample_words, _sample_tags = train_full[0]
    _check_ds = to_hf_dataset([train_full[0]], _tok)
    _enc_check = _tok(_sample_words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    _word_ids_check = _enc_check.word_ids()
    _labels_check = _check_ds[0]["labels"]

    assert len(_labels_check) == len(_word_ids_check)
    _labeled = set(label_positions(_word_ids_check, _enc_check.tokens()).values())
    for _pos, _wid in enumerate(_word_ids_check):
        if _wid is None:
            assert _labels_check[_pos] == -100, "special token must be -100"
        elif _pos in _labeled:
            assert _labels_check[_pos] == label2id[_sample_tags[_wid]], "labeled subword must carry the word's tag"
        else:
            assert _labels_check[_pos] == -100, "other subwords must be -100"

    print(f"label-alignment self-test ({_model_key}, {_checkpoint}): PASSED "
          f"({len(_sample_words)} words -> {len(_word_ids_check)} subword positions)")

    # A standalone "▁" should only carry a label if the word has no other piece.
    _words_total = _bare_first = _bare_labeled = 0
    for _words, _ in make_subset(train_full, min(2000, len(train_full))):
        _enc = _tok(_words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
        _pieces, _ids = _enc.tokens(), _enc.word_ids()
        _bare_first += sum(_pieces[_ids.index(_w)] == "▁" for _w in set(_ids) - {None})
        for _p in label_positions(_ids, _pieces).values():
            _words_total += 1
            _bare_labeled += _pieces[_p] == "▁"
    print(f"  standalone '▁' as first subword: {_bare_first} / {_words_total} words; "
          f"still carrying the label: {_bare_labeled}")



## Prediction and scoring

Same evaluator as every other phase (`src.eval.evaluate`, entity-level
`seqeval` micro-F1). Predictions are mapped back from subwords to words the
same way the labels were built: only each word's labeled subword is read,
and every other position is skipped. Words lost to
truncation stay in the gold labels and count as predicted `O`, so a truncated
entity is scored as a miss.

In [ ]:
def predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    dev = next(model.parameters()).device
    y_true, y_pred = [], []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        words_batch = [r[0] for r in chunk]
        tags_batch = [r[1] for r in chunk]
        enc = tokenizer(words_batch, is_split_into_words=True, truncation=True, max_length=max_length,
                         padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            logits = model(**enc).logits
        preds = logits.argmax(-1).cpu().numpy()
        for b, tags in enumerate(tags_batch):
            pred_tags = ["O"] * len(tags)
            for wid, pos in label_positions(enc.word_ids(batch_index=b), enc.tokens(batch_index=b)).items():
                pred_tags[wid] = id2label[int(preds[b][pos])]
            y_pred.append(pred_tags)
            y_true.append(list(tags))
    return y_true, y_pred


for _model_key, _checkpoint in MODELS.items():
    _tok = AutoTokenizer.from_pretrained(_checkpoint)
    _long = sum(1 for words, _ in test if len(_tok(words, is_split_into_words=True)["input_ids"]) > MAX_LENGTH)
    print(f"{_model_key}: gold test sentences with > {MAX_LENGTH} subword tokens "
          f"(tail words count as predicted O): {_long} / {len(test)}")


## Reusable training run

One function, called once per run. It samples the subset, checks entity
density, fine-tunes with HuggingFace `Trainer` (checkpointing every epoch so
a Kaggle disconnect only costs the current step), and scores on validation
and the full gold test set.

The order after training is fixed: **upload the final model and tokenizer,
confirm the upload, log one `results.csv` row with the pinned link, and only
then delete the local copy.** If the upload fails, the run stops and the local
folder stays. `run_name` contains a config hash, so a run can only resume from
a checkpoint made with identical settings.

In [ ]:
import gc
import hashlib
import json
import math
import shutil

import transformers
from huggingface_hub import HfApi
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

hf_api = HfApi()
HF_NAMESPACE = hf_api.whoami()["name"]   # fails here, before any training, if HF_TOKEN is missing
RUNS = []   # MuRIL runs that count towards the curve and the size choice


class ValF1Callback(TrainerCallback):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.history = []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        f1 = evaluate(*predict_tags(model, self.tokenizer, val))["F1_overall"]
        self.history.append(f"{f1:.4f}")
        print(f"epoch {state.epoch:.0f}: val F1 {f1:.4f}")
        model.train()


def publish_checkpoint(ckpt_dir, run_name):
    repo_id = f"{HF_NAMESPACE}/naamrekha-{run_name}"
    hf_api.create_repo(repo_id, private=HF_PRIVATE, exist_ok=True)
    commit = hf_api.upload_folder(repo_id=repo_id, folder_path=ckpt_dir,
                                  ignore_patterns=["checkpoint-*", "runs/*"])
    remote = {f.rfilename: f.size for f in hf_api.model_info(repo_id, revision=commit.oid, files_metadata=True).siblings}
    local_size = os.path.getsize(os.path.join(ckpt_dir, "model.safetensors"))
    if remote.get("model.safetensors") != local_size:
        raise RuntimeError(f"upload to {repo_id} not confirmed; keeping {ckpt_dir}")
    return repo_id, commit.oid


def train_run(model_key, subset_size, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=SEED, resume=True):
    checkpoint = MODELS[model_key]
    random.seed(seed)
    torch.manual_seed(seed)

    train = make_subset(train_full, subset_size, seed)
    entity_report(train, f"{model_key} train subset n={len(train)}")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForTokenClassification.from_pretrained(
        checkpoint, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )
    train_ds = to_hf_dataset(train, tokenizer)
    collator = DataCollatorForTokenClassification(tokenizer)

    # CUDA_VISIBLE_DEVICES=0 (Config cell) keeps this at exactly one GPU, so
    # batch_size below is the real effective batch size, not silently doubled
    # by DataParallel across Kaggle's two T4s.
    n_gpu = max(torch.cuda.device_count(), 1)
    total_steps = math.ceil(len(train_ds) / (batch_size * n_gpu)) * epochs
    warmup_steps = int(0.1 * total_steps)

    config = {"checkpoint": checkpoint, "lang": LANG, "n": len(train), "epochs": epochs, "lr": lr,
              "batch_size": batch_size, "seed": seed, "max_length": MAX_LENGTH, "gpus": n_gpu,
              "warmup_steps": warmup_steps}
    config_hash = hashlib.sha1(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]
    run_name = (f"{model_key.lower()}_{LANG}_n{len(train)}_e{epochs}_lr{lr}_s{seed}_{config_hash}"
                + ("_debug" if DEBUG else ""))
    ckpt_dir = os.path.join(STORAGE, "phase3", run_name)
    has_ckpt = os.path.isdir(ckpt_dir) and any(f.startswith("checkpoint-") for f in os.listdir(ckpt_dir))

    args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        fp16=torch.cuda.is_available(),
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        seed=seed,
        report_to=[],
    )
    val_callback = ValF1Callback(tokenizer)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator,
                       processing_class=tokenizer, callbacks=[val_callback])
    trainer.train(resume_from_checkpoint=has_ckpt and resume)
    trainer.save_model(ckpt_dir)

    val_f1 = evaluate(*predict_tags(model, tokenizer, val))["F1_overall"]
    y_true, y_pred = predict_tags(model, tokenizer, test)
    scores = evaluate(y_true, y_pred)

    repo_id, revision = publish_checkpoint(ckpt_dir, run_name)
    checkpoint_url = f"https://huggingface.co/{repo_id}/tree/{revision}"

    notes = (f"seed={seed}; lr={lr}; bs={batch_size}; effective_bs={batch_size * n_gpu}; gpus={n_gpu}; "
             f"warmup_steps={warmup_steps}; epochs={epochs}; max_len={MAX_LENGTH}; checkpoint={checkpoint}; "
             f"config={config_hash}; val_F1={val_f1:.4f}; val_F1_by_epoch={'/'.join(val_callback.history)}; "
             f"git={GIT_COMMIT}; torch={torch.__version__}; transformers={transformers.__version__}"
             + ("; DEBUG run -- not a real result" if DEBUG else ""))
    log_result(model=model_key, train_lang=LANG, test_lang=LANG, subset_size=len(train),
                epochs=epochs, scores=scores, checkpoint_path=checkpoint_url, notes=notes)

    with open(os.path.join(REPO_ROOT, "results", "results.csv"), encoding="utf-8") as f:
        print(f.read().splitlines()[-1])

    shutil.rmtree(ckpt_dir)
    print(f"published {checkpoint_url} and removed the local copy")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return {"model_key": model_key, "subset_size": len(train), "epochs": epochs, "lr": lr, "seed": seed,
            "val_f1": val_f1, "scores": scores, "repo_id": repo_id, "revision": revision,
            "checkpoint_url": checkpoint_url}



## Pipeline check (`DEBUG = True`)

Runs the whole pipeline once on a tiny CPU-sized slice. The score is not
meaningful (too little data, 1 epoch) -- it is logged with `DEBUG` in
`notes` so `scripts/phase3_gate.py` and the scaling-curve plot both ignore
it. This is only proof that label alignment, training, prediction, scoring,
upload, reload, and logging all work end-to-end before spending Kaggle GPU
hours on it.

In [ ]:
if DEBUG:
    RUNS.append(train_run("MuRIL", SUBSET_SIZES[0]))
    print("DEBUG scores (meaningless at this size):", RUNS[-1]["scores"])


## Scaling curve (run on Kaggle, `DEBUG = False`)

Runs, in order:

1. **10k diagnosis:** 9 epochs at the same LR, to test whether the 3-epoch
   collapse is under-training. If it still collapses, one more run at LR 5e-5.
2. **Curve:** MuRIL at each size in `SUBSET_SIZES`, largest last.
3. **Extra seeds:** `SEEDED_SIZES` on `EXTRA_SEEDS`.
4. **Equal-steps check:** 30k for 10 epochs, about the same step count as 100k
   for 3 epochs.
5. **Size choice:** the rule in the Config cell, on validation F1. If the chosen
   size has fewer than three seeds, its missing seeds run before it is accepted.

In [ ]:
from collections import defaultdict
from statistics import mean


def choose_subset_size(runs):
    val_by_size = defaultdict(list)
    for r in runs:
        val_by_size[r["subset_size"]].append(r["val_f1"])
    means = {n: mean(f1s) for n, f1s in val_by_size.items()}
    best = max(means.values())
    for n in sorted(val_by_size):
        if min(val_by_size[n]) >= COLLAPSE_F1 and means[n] >= best - SIZE_TOLERANCE:
            return n
    raise RuntimeError(f"no subset size qualifies: {dict(val_by_size)}")


if not DEBUG:
    diagnosis = train_run("MuRIL", 10_000, epochs=9)
    if diagnosis["val_f1"] < COLLAPSE_F1:
        train_run("MuRIL", 10_000, epochs=9, lr=5e-5)

    for _n in SUBSET_SIZES:
        RUNS.append(train_run("MuRIL", _n))
    for _seed in EXTRA_SEEDS:
        for _n in SEEDED_SIZES:
            RUNS.append(train_run("MuRIL", _n, seed=_seed))

    train_run("MuRIL", 30_000, epochs=10)

    while True:
        _chosen = choose_subset_size(RUNS)
        _done = {r["seed"] for r in RUNS if r["subset_size"] == _chosen}
        _missing = [s for s in [SEED, *EXTRA_SEEDS] if s not in _done]
        if not _missing:
            break
        for _seed in _missing:
            RUNS.append(train_run("MuRIL", _chosen, seed=_seed))



## Plot: F1 vs. training size

Reads the logged MuRIL `hi` -> `hi` rows straight from `results/results.csv`,
so the plot is always built from what is actually logged. Only rows from this
code commit with the standard epochs and LR are plotted, so old-setup runs
and the diagnostic runs cannot enter the curve. Points show the mean across
seeds, and bars show the min-to-max range.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_scaling_curve():
    results_df = pd.read_csv(os.path.join(REPO_ROOT, "results", "results.csv"))
    curve_rows = results_df[
        (results_df["model"] == "MuRIL")
        & (results_df["train_lang"] == LANG)
        & (results_df["test_lang"] == LANG)
        & (results_df["epochs"] == EPOCHS)
        & (results_df["notes"].str.contains(f"lr={LR};", regex=False, na=False))
        & (results_df["notes"].str.contains(f"git={GIT_COMMIT};", regex=False, na=False))
        & (~results_df["notes"].str.contains("DEBUG", na=False))
    ]
    curve = curve_rows.groupby("subset_size")["F1_overall"].agg(["mean", "min", "max", "count"]).reset_index()

    if len(curve) == 0:
        print("No real (non-DEBUG) MuRIL rows logged yet -- run the scaling-curve cell on Kaggle first.")
    else:
        baseline_f1 = results_df.loc[
            (results_df["model"] == "BiLSTM-CRF") & (~results_df["notes"].str.contains("DEBUG", na=False)),
            "F1_overall",
        ].max()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.errorbar(curve["subset_size"], curve["mean"],
                    yerr=[curve["mean"] - curve["min"], curve["max"] - curve["mean"]],
                    marker="o", capsize=3, label="MuRIL (hi gold F1, mean and range over seeds)")
        ax.axhline(baseline_f1, color="gray", linestyle="--", label=f"BiLSTM-CRF baseline ({baseline_f1:.4f})")
        ax.set_xscale("log")
        ax.set_xlabel("training subset size (sentences, log scale)")
        ax.set_ylabel("Hindi gold entity F1 (overall)")
        ax.set_title("Phase 3: MuRIL scaling curve")
        ax.legend()
        fig.tight_layout()
        fig_path = os.path.join(REPO_ROOT, "results", "phase3_scaling_curve.png")
        fig.savefig(fig_path, dpi=150)
        print("saved", fig_path)
        print(curve.to_string(index=False))


# A plot failure must not stop the IndicBERT and handoff cells; it can be rebuilt from results.csv.
try:
    plot_scaling_curve()
except Exception as exc:
    print(f"WARNING: scaling plot failed ({exc!r})")


## Choose the working subset size

Chosen by rule on **validation** F1, not by eye and not from the gold test
set: the smallest size with no collapsed seed whose mean validation F1 is
within `SIZE_TOLERANCE` of the best size. If the curve is still climbing
steeply at the largest size a Kaggle session allows, say so in
[docs/phase3/README.md](../docs/phase3/README.md) rather than forcing a
plateau that isn't there.

In [ ]:
CHOSEN_SUBSET_SIZE = SUBSET_SIZES[0] if DEBUG else choose_subset_size(RUNS)
HANDOFF_RUN = next(r for r in RUNS if r["subset_size"] == CHOSEN_SUBSET_SIZE and r["seed"] == SEED)
print("working subset size:", CHOSEN_SUBSET_SIZE)
print("handoff checkpoint:", HANDOFF_RUN["checkpoint_url"])


## IndicBERT comparison (run on Kaggle, after choosing the size above)

One run at `CHOSEN_SUBSET_SIZE`, same harness, so MuRIL and IndicBERT are
compared at the same training size.

In [ ]:
indicbert_run = train_run("IndicBERT", CHOSEN_SUBSET_SIZE)
print("IndicBERT scores:", indicbert_run["scores"])
print("IndicBERT checkpoint:", indicbert_run["checkpoint_url"])



## Verify the Phase 4 handoff checkpoint

Reloads the chosen MuRIL run from its pinned Hugging Face revision, re-scores
Hindi gold, and stops if the score does not match the logged row. Phase 4
should load this exact revision.

In [ ]:
_repo_id, _revision = HANDOFF_RUN["repo_id"], HANDOFF_RUN["revision"]
_model = AutoModelForTokenClassification.from_pretrained(_repo_id, revision=_revision).to(device)
_tok = AutoTokenizer.from_pretrained(_repo_id, revision=_revision)
_rescored = evaluate(*predict_tags(_model, _tok, test))["F1_overall"]
_logged = HANDOFF_RUN["scores"]["F1_overall"]
if abs(_rescored - _logged) > 1e-4:
    raise RuntimeError(f"reloaded checkpoint scores {_rescored:.4f}, logged row says {_logged:.4f}")
print(f"handoff verified: {_repo_id}@{_revision} re-scored {_rescored:.4f} (logged {_logged:.4f})")



## Gate

Run from the repo root (locally -- this only reads committed files, no GPU
needed):

```
uv run python scripts/phase3_gate.py
```

It checks: every Phase 3 row comes from one top-to-bottom execution of one
commit, uses one GPU and warmup, and links to a pinned checkpoint; the chosen
size follows the validation-F1 rule and is confirmed on three seeds; the
handoff model was reloaded, re-scored, and clearly beats the Phase 2 baseline
(0.8038); an IndicBERT row exists at the chosen size; and the README, roadmap,
and decisions log agree with the result.